# チャンク化とベクトル化

このノートブックでは、RAGシステムの基礎となる **チャンク化** と **ベクトル化** の関係を実際に体験します。

## 流れ
```
文書テキスト
  ↓ チャンク化（テキストを小さな塊に分割）
テキストの断片リスト
  ↓ ベクトル化（意味を数値に変換）
数値ベクトルのリスト
  ↓ 類似度計算（コサイン類似度）
意味的に近いチャンクを検索
```

## 0. ライブラリ・フォントのインストール

In [ ]:
!pip install sentence-transformers matplotlib scikit-learn numpy japanize-matplotlib -q
print('インストール完了！')

---
## ステップ1: サンプル文書の準備

今回は「機械学習」についての説明文を使います。

In [ ]:
document = """
機械学習は、コンピュータがデータから自動的に学習する技術です。
従来のプログラミングでは、人間がルールを明示的に記述する必要がありました。
しかし機械学習では、大量のデータを与えることでモデルが自らパターンを発見します。

機械学習には主に3つの学習方法があります。
教師あり学習は、正解ラベル付きのデータを使ってモデルを訓練します。
教師なし学習は、ラベルなしデータからパターンやクラスタを発見します。
強化学習は、試行錯誤を通じて報酬を最大化する方策を学びます。

ディープラーニングは機械学習の一分野で、多層のニューラルネットワークを使います。
画像認識、音声認識、自然言語処理など幅広い分野で優れた性能を発揮しています。
GPUの発展と大規模データの普及により、ディープラーニングは急速に進化しました。

自然言語処理は、コンピュータが人間の言語を理解・生成する技術です。
テキスト分類、機械翻訳、質問応答システムなどに活用されています。
近年はTransformerアーキテクチャが主流となり、ChatGPTのような大規模言語モデルが登場しました。
"""

print("=== サンプル文書 ===")
print(document)
print(f"\n文書の文字数: {len(document)}文字")

---
## ステップ2: チャンク化

文書を小さな塊（チャンク）に分割します。
ここでは**段落ごと**に分割する方法を使います。

In [ ]:
def chunk_by_paragraph(text):
    """段落ごとにチャンク化する（空行で区切る）"""
    paragraphs = text.strip().split('\n\n')
    chunks = [p.strip().replace('\n', ' ') for p in paragraphs if p.strip()]
    return chunks

chunks = chunk_by_paragraph(document)

print(f"=== チャンク化結果: {len(chunks)}個のチャンク ===")
for i, chunk in enumerate(chunks):
    print(f"\n[チャンク {i+1}] ({len(chunk)}文字)")
    print(chunk)

### 比較: 固定長チャンク化

もう一つの方法として、**文字数で機械的に分割**する方法も見てみましょう。

In [ ]:
def chunk_by_fixed_size(text, chunk_size=80, overlap=20):
    """固定文字数でチャンク化（オーバーラップあり）"""
    text = text.replace('\n', ' ').strip()
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

fixed_chunks = chunk_by_fixed_size(document)

print(f"=== 固定長チャンク化: {len(fixed_chunks)}個 ===")
for i, chunk in enumerate(fixed_chunks[:4]):
    print(f"[チャンク {i+1}]: {chunk}")

print("\n→ 段落チャンクの方が意味的なまとまりを保てることが分かります。")

---
## ステップ3: ベクトル化

チャンクを数値ベクトルに変換します。
`sentence-transformers` の多言語モデルを使います。

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 多言語対応モデルをロード
print("モデルをロード中...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("ロード完了!")

# チャンクをベクトル化
embeddings = model.encode(chunks)

print(f"\n=== ベクトル化結果 ===")
print(f"チャンク数: {len(chunks)}")
print(f"各ベクトルの次元数: {embeddings.shape[1]}")
print(f"\n[チャンク1のベクトル（最初の10次元）]")
print(np.round(embeddings[0][:10], 4))
print("...（合計384次元）")

---
## ステップ4: 類似度検索

クエリ（質問文）をベクトル化して、意味的に近いチャンクを検索します。
これがRAGの「検索」部分です。

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query, chunks, embeddings, top_k=2):
    """クエリに最も近いチャンクを返す"""
    query_vec = model.encode([query])
    similarities = cosine_similarity(query_vec, embeddings)[0]
    ranked = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

    print(f"クエリ: '{query}'")
    print("─" * 50)
    for rank, (idx, score) in enumerate(ranked[:top_k], 1):
        print(f"【第{rank}位】 類似度: {score:.4f}")
        print(f"{chunks[idx]}")
        print()

# クエリ1: ニューラルネットワークについて
print("=" * 50)
search("ニューラルネットワークについて教えて", chunks, embeddings)

print("=" * 50)
# クエリ2: ChatGPTについて
search("ChatGPTはどんな技術を使っている？", chunks, embeddings)

print("=" * 50)
# クエリ3: 学習の種類
search("機械学習の学習方法の種類は？", chunks, embeddings)

---
## ステップ5: ベクトルの可視化

高次元ベクトルをPCAで2次元に圧縮して、チャンク同士の「意味的な距離」を可視化します。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import japanize_matplotlib  # noqa: F401 — これだけで日本語が使えるようになる
from sklearn.decomposition import PCA

# クエリもベクトル化して一緒に可視化
queries = ["ニューラルネットワーク", "ChatGPT", "学習方法の種類"]
query_vecs = model.encode(queries)

all_vecs = np.vstack([embeddings, query_vecs])
labels = [f"チャンク{i+1}" for i in range(len(chunks))] + queries
colors = ['#4A90D9'] * len(chunks) + ['#E74C3C'] * len(queries)
markers = ['o'] * len(chunks) + ['*'] * len(queries)

# PCAで2次元に圧縮
pca = PCA(n_components=2)
reduced = pca.fit_transform(all_vecs)

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor('#F8F9FA')
fig.patch.set_facecolor('white')

for i, (x, y) in enumerate(reduced):
    ax.scatter(x, y, c=colors[i], s=300 if markers[i]=='*' else 200,
               marker=markers[i], zorder=3, edgecolors='white', linewidths=1.5)
    ax.annotate(labels[i], (x, y),
                textcoords='offset points', xytext=(8, 8),
                fontsize=10, color='#2C3E50',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#BDC3C7', alpha=0.8))

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#4A90D9', markersize=12, label='チャンク（文書断片）'),
    Line2D([0], [0], marker='*', color='w', markerfacecolor='#E74C3C', markersize=15, label='クエリ（検索語）')
]
ax.legend(handles=legend_elements, loc='best', fontsize=10)

ax.set_title('チャンクとクエリのベクトル空間（PCAで2次元圧縮）\n近い点 = 意味が似ている', fontsize=13, pad=15)
ax.set_xlabel('第1主成分', fontsize=10)
ax.set_ylabel('第2主成分', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('vector_space.png', dpi=150, bbox_inches='tight')
plt.show()
print("→ クエリ（赤★）が意味的に近いチャンク（青●）の近くに配置されているのが分かります")

---
## ステップ6: チャンク間の類似度ヒートマップ

全チャンク同士の類似度を可視化します。

In [ ]:
import matplotlib.font_manager as fm
import matplotlib
import japanize_matplotlib  # noqa: F401

sim_matrix = cosine_similarity(embeddings)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)

chunk_labels = [f'チャンク{i+1}' for i in range(len(chunks))]
ax.set_xticks(range(len(chunks)))
ax.set_yticks(range(len(chunks)))
ax.set_xticklabels(chunk_labels, fontsize=10)
ax.set_yticklabels(chunk_labels, fontsize=10)

for i in range(len(chunks)):
    for j in range(len(chunks)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                fontsize=11, color='white' if sim_matrix[i,j] > 0.6 else '#2C3E50', fontweight='bold')

plt.colorbar(im, ax=ax, label='コサイン類似度')
ax.set_title('チャンク間のコサイン類似度\n（1.0 = 完全一致、値が高いほど意味が近い）', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nチャンクの内容:')
for i, chunk in enumerate(chunks):
    print(f'  チャンク{i+1}: {chunk[:40]}...')

---
## まとめ

| ステップ | 処理 | 入力 | 出力 |
|------|------|------|------|
| チャンク化 | テキストを分割 | 長い文書 | テキストの断片リスト |
| ベクトル化 | 意味を数値に変換 | テキストの断片 | 384次元のベクトル |
| 類似度計算 | コサイン類似度 | 2つのベクトル | 0〜1のスコア |
| 検索 | スコア上位を取得 | クエリ+全ベクトル | 関連チャンク |

**チャンク化とベクトル化は別の処理ですが、チャンク化してからベクトル化することで、意味的な検索が可能になります。**

これがRAG（Retrieval-Augmented Generation）の基本的な仕組みです